# Huấn luyện Mô hình CRAM-ABSA v2 Sinh Nhận Xét Tự Động 100% Bằng Neural Seq2Seq (modelv2.pt)


In [ ]:
import os
import re
import sys
import math
import json
import time
import shutil
import pickle
import random
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    get_cosine_schedule_with_warmup
)
from transformers.modeling_outputs import BaseModelOutput
from sklearn.metrics import f1_score, mean_absolute_error, accuracy_score, cohen_kappa_score

import nltk
for pkg in ["stopwords", "punkt", "averaged_perceptron_tagger", "wordnet"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg in ["stopwords", "wordnet"] else f"tokenizers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"PyTorch: {torch.__version__} | Device: {DEVICE} | Số lượng GPU khả dụng: {NUM_GPUS}")


## 1. Thiết lập Cấu hình Huấn luyện CRAM-ABSA v2


In [ ]:
class TrainConfigV2:
    PARQUET_CANDIDATES = [
        "/kaggle/input/dap_dts/hotel_reviews.parquet",
        "/kaggle/input/dap-dts/hotel_reviews.parquet",
        "/kaggle/input/hotel-reviews-cram-absa/hotel_reviews.parquet",
        "/kaggle/input/tripadvisor-hotel-reviews/hotel_reviews.parquet",
        "data/df/hotel_reviews.parquet",
        "hotel_reviews.parquet"
    ]
    OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "model")
    LOCAL_MODEL_DIR = Path("model")

    MODEL_FILENAME = "modelv2.pt"
    MODEL_SAVE_PATH = OUTPUT_DIR / "modelv2.pt"
    LOCAL_MODEL_SAVE_PATH = LOCAL_MODEL_DIR / "modelv2.pt"
    HISTORY_SAVE_PATH = OUTPUT_DIR / "training_history_v2.json"

    BACKBONE_NAME = "bert-base-uncased"
    GEN_MODEL_NAME = "google/flan-t5-small"
    HIDDEN_DIM = 768
    NUM_FEATURES = 20
    PROJ_DIM = 256
    ATTN_HEADS = 8
    NUM_SOFT_PROMPT_TOKENS = 4
    CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
    NUM_ASPECTS = 6
    NUM_SENTIMENT_CLASSES = 4
    NUM_STARS = 5
    MAX_SEQ_LEN = 256
    MAX_COMMENT_LEN = 48

    PER_GPU_BATCH_SIZE = 16
    BATCH_SIZE = PER_GPU_BATCH_SIZE * max(1, NUM_GPUS)
    GRAD_ACCUM_STEPS = 2
    NUM_WORKERS = 2 if os.name != "nt" else 0
    PIN_MEMORY = torch.cuda.is_available()
    USE_AMP_FP16 = torch.cuda.is_available()

    EPOCHS = 8
    LR_BACKBONE = 3e-5
    LR_HEADS = 1e-4
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.10
    LABEL_SMOOTHING = 0.08
    MAX_GRAD_NORM = 1.0

    BETA_CORN = 0.30
    BETA_OVERLAP = 0.18
    BETA_RATIONALE = 0.25
    BETA_COMMENTARY = 0.35

    FEATURE_COLS_20 = [
        "feat_aspect_count",
        "feat_dominant_aspect",
        "feat_net_sentiment_score",
        "feat_sentiment_conflict",
        "feat_evidence_density",
        "feat_avg_rationale_len",
        "feat_has_cross_overlap",
        "feat_service_sentiment",
        "feat_facility_sentiment",
        "feat_experience_sentiment",
        "feat_amenity_sentiment",
        "feat_loyalty_sentiment",
        "feat_branding_sentiment",
        "feat_aspect_coverage_ratio",
        "feat_recency_weight",
        "feat_consensus_strength",
        "feat_temporal_aspect_sentiment",
        "feat_nltk_clean_token_ratio",
        "feat_nltk_opinion_negation_density",
        "feat_ebr_aspect_prototype_sim"
    ]

cfg = TrainConfigV2()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model v2 save path: {cfg.MODEL_SAVE_PATH}")


## 2. Tiền xử lý Dual-Stream NLTK và Trích xuất 20 Đặc trưng Đầu vào


In [ ]:
try:
    from nltk.corpus import stopwords
    RAW_NLTK_STOPWORDS = set(stopwords.words("english"))
except Exception:
    RAW_NLTK_STOPWORDS = {
        "a", "an", "the", "and", "or", "if", "in", "on", "at", "to", "from", "by", "for",
        "with", "about", "as", "into", "like", "through", "after", "over", "between", "out",
        "of", "during", "before", "under", "around", "among", "is", "are", "was", "were",
        "be", "been", "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "shall", "should", "can", "could", "may", "might", "must", "i", "me", "my", "myself",
        "we", "our", "ours", "ourselves", "you", "your", "yours", "he", "him", "his", "she",
        "her", "hers", "it", "its", "they", "them", "their", "theirs", "this", "that", "these",
        "those", "am", "so", "than", "too", "very", "just", "where", "when", "who", "whom",
        "which", "what", "there", "here", "all", "any", "both", "each", "few", "more", "most",
        "other", "some", "such", "only", "own", "same", "then", "once"
    }

CONTRACTIONS_MAP = {
    "can't": "cannot", "won't": "will not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am"
}

NEGATION_AND_CONTRAST_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nothing", "nowhere", "hardly",
    "barely", "scarcely", "without", "but", "however", "although", "though", "yet",
    "despite", "unfortunately", "except", "fault"
}

SAFE_STOPWORDS = RAW_NLTK_STOPWORDS - NEGATION_AND_CONTRAST_WORDS

OPINION_LEXICON_POS = {
    "good", "great", "excellent", "amazing", "wonderful", "friendly", "helpful", "clean",
    "spacious", "comfortable", "delicious", "lovely", "perfect", "best", "nice", "fantastic",
    "superb", "polite", "attentive", "convenient", "beautiful", "cozy", "relaxing", "awesome",
    "recommend", "recommended", "happy", "pleased", "impressed", "welcoming", "professional",
    "quiet", "modern", "fresh", "tasty", "worth", "luxury", "luxurious", "outstanding", "warm"
}

OPINION_LEXICON_NEG = {
    "bad", "poor", "terrible", "horrible", "awful", "slow", "rude", "dirty", "noisy", "small",
    "old", "outdated", "broken", "cold", "hard", "uncomfortable", "overpriced", "expensive",
    "worst", "disappointed", "disappointing", "unfriendly", "unhelpful", "smell", "smelly",
    "stain", "stained", "cockroach", "bug", "mold", "issue", "problem", "complaint", "wait",
    "waiting", "delayed", "mediocre", "boring", "crowded", "weak", "fault"
}

HEDGING_MODIFIERS = {
    "moderate", "okay", "ok", "decent", "fair", "acceptable", "average", "slightly",
    "bit", "ordinary", "normal", "passable", "tolerable", "adequate", "fine", "basic"
}

INTENSIFIERS = {
    "very", "extremely", "really", "super", "highly", "totally", "absolutely", "completely",
    "quite", "pretty", "so", "too", "exceptionally", "incredibly", "amazingly", "remarkably"
}

ASPECT_KEYWORDS = {
    "Service": {
        "staff", "service", "services", "member", "members", "reception", "receptionist",
        "front", "desk", "manager", "waiter", "waitress", "concierge", "bellboy", "helpful",
        "friendly", "polite", "check", "checkin", "checkout", "attendant", "housekeeping",
        "host", "crew", "hospitality", "team", "assistance", "welcoming", "professional", "attentive"
    },
    "Facility": {
        "facility", "facilities", "maintenance", "traffic", "location", "building", "property",
        "room", "rooms", "bed", "beds", "bathroom", "shower", "toilet", "air", "ac",
        "conditioning", "clean", "cleanliness", "noise", "noisy", "soundproof", "view",
        "balcony", "floor", "elevator", "lift", "pillow", "mattress", "furniture", "decor",
        "water", "carpet", "lobby", "parking", "surrounding", "area"
    },
    "Experience": {
        "stay", "stayed", "experience", "holiday", "vacation", "trip", "night", "nights",
        "overall", "atmosphere", "vibe", "relax", "relaxing", "enjoy", "enjoyed", "memorable",
        "comfort", "comfortable", "quiet", "peaceful", "feeling", "impression", "journey", "moment"
    },
    "Amenity": {
        "kitchen", "food", "foods", "meal", "meals", "dish", "dishes", "taste", "tasty",
        "delicious", "flavor", "dining", "breakfast", "buffet", "pool", "swimming", "spa",
        "gym", "fitness", "wifi", "internet", "restaurant", "bar", "drink", "drinks",
        "coffee", "tea", "lounge", "shuttle", "massage", "rooftop", "menu", "cuisine"
    },
    "Loyalty": {
        "back", "return", "returned", "returning", "again", "times", "visit", "visits",
        "visited", "revisit", "recommend", "recommended", "definitely", "choose", "future",
        "next", "repeat", "loyal", "regular", "choice", "staying"
    },
    "Branding": {
        "star", "stars", "luxury", "luxurious", "brand", "chain", "standard", "reputation",
        "class", "five", "four", "management", "value", "price", "pricing", "worth",
        "prestigious", "accor", "sofitel", "pullman", "novotel", "marriott", "hilton",
        "intercontinental", "sheraton", "flagship", "resort"
    }
}

ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(cfg.CANONICAL_ASPECTS)}
ASPECT_ID_MAP["None"] = 0
SENTIMENT_TO_ID = {"Negative": 1, "Neutral": 2, "Positive": 3}
ID_TO_SENTIMENT = {0: "Not Mentioned", 1: "Negative", 2: "Neutral", 3: "Positive"}

def nltk_negation_aware_preprocess(text: str) -> Tuple[List[str], List[str]]:
    lower_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        lower_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, lower_txt)
        lower_txt = lower_txt.replace(k, v)
    lower_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", lower_txt)
    raw_tokens = re.findall(r"[a-z]+", lower_txt)
    clean_tokens = [t for t in raw_tokens if t not in SAFE_STOPWORDS and len(t) > 1]
    return raw_tokens, clean_tokens

def preprocess_text_paper1(text: str) -> Tuple[str, str]:
    norm_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        norm_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, norm_txt)
        norm_txt = norm_txt.replace(k, v)
    norm_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", norm_txt)
    clean_review_text = re.sub(r"[^a-z0-9.,!?'\s-]", " ", norm_txt)
    clean_review_text = re.sub(r"\s+", " ", clean_review_text).strip()
    _, clean_tokens = nltk_negation_aware_preprocess(norm_txt)
    paper1_nltk_text = " ".join(clean_tokens)
    return clean_review_text, paper1_nltk_text

def parse_annotations_safe(anns: Any) -> List[Dict[str, Any]]:
    if anns is None:
        return []
    if isinstance(anns, np.ndarray):
        anns = anns.tolist()
    elif isinstance(anns, str):
        try:
            anns = json.loads(anns)
        except Exception:
            return []
    if not isinstance(anns, list):
        return []
    return [dict(a) for a in anns if isinstance(a, dict)]

def extract_20_features_for_new_review(review_text: str, recency_weight: float = 1.0) -> np.ndarray:
    raw_text = unicodedata.normalize("NFC", str(review_text).strip())
    char_len = max(len(raw_text), 1)
    raw_tokens, clean_tokens = nltk_negation_aware_preprocess(raw_text)
    total_raw_toks = max(len(raw_tokens), 1)

    clean_token_ratio = float(min(len(clean_tokens) / total_raw_toks, 1.0))
    opinion_neg_cnt = sum(
        1 for t in raw_tokens
        if (t in OPINION_LEXICON_POS or t in OPINION_LEXICON_NEG or t in NEGATION_AND_CONTRAST_WORDS or t in INTENSIFIERS)
    )
    opinion_neg_density = float(min(opinion_neg_cnt / total_raw_toks, 1.0))

    tok_counts = Counter(clean_tokens)
    aspect_sims = []
    for asp in cfg.CANONICAL_ASPECTS:
        kw_set = ASPECT_KEYWORDS[asp]
        overlap_w = sum(cnt for tok, cnt in tok_counts.items() if tok in kw_set)
        norm_sim = overlap_w / math.sqrt(len(clean_tokens) * len(kw_set)) if clean_tokens else 0.0
        aspect_sims.append(min(1.0, norm_sim * 3.5))
    top_sims = sorted(aspect_sims, reverse=True)[:3]
    ebr_proto_sim = float(sum(top_sims) / len(top_sims)) if top_sims else 0.0

    clauses = [
        p.strip()
        for sent in re.split(r"[.!?]+", raw_text) if sent.strip()
        for p in re.split(r"\s+(?:and|but|however|while|although|though)\s+|\s*;\s*", sent, flags=re.IGNORECASE)
        if p.strip()
    ]
    if not clauses and raw_text:
        clauses = [raw_text]

    aspect_scores = {}
    matched_clauses_per_aspect = defaultdict(list)
    overlap_flag = 0

    for cl in clauses:
        s_raw, s_clean = nltk_negation_aware_preprocess(cl)
        matched_asps = [asp for asp in cfg.CANONICAL_ASPECTS if any(t in ASPECT_KEYWORDS[asp] for t in s_raw)]
        if len(matched_asps) >= 2:
            overlap_flag = 1
        pos_hits = sum(1 for t in s_raw if t in OPINION_LEXICON_POS)
        neg_hits = sum(1 for t in s_raw if t in OPINION_LEXICON_NEG)
        hedge_hits = sum(1 for t in s_raw if t in HEDGING_MODIFIERS)
        for idx_t, tok in enumerate(s_raw[:-1]):
            if tok in {"not", "never", "no", "hardly"}:
                nxt = s_raw[idx_t + 1]
                if nxt in OPINION_LEXICON_POS:
                    pos_hits = max(0, pos_hits - 1)
                    neg_hits += 1
                elif nxt in OPINION_LEXICON_NEG:
                    neg_hits = max(0, neg_hits - 1)
                    pos_hits += 1
        sent_pol = (pos_hits - neg_hits - 0.25 * hedge_hits) / max(pos_hits + neg_hits + hedge_hits, 1) if (pos_hits + neg_hits + hedge_hits) > 0 else 0.25
        for asp in matched_asps:
            matched_clauses_per_aspect[asp].append((cl, sent_pol))

    for asp in cfg.CANONICAL_ASPECTS:
        items = matched_clauses_per_aspect.get(asp, [])
        aspect_scores[asp] = float(np.clip(np.mean([sc for _, sc in items]), -1.0, 1.0)) if items else 0.0

    active_aspects = [asp for asp in cfg.CANONICAL_ASPECTS if matched_clauses_per_aspect.get(asp)]
    asp_count = len(active_aspects)
    dominant_aspect = ASPECT_ID_MAP.get(active_aspects[0], 0) if active_aspects else 0
    active_vals = [aspect_scores[a] for a in active_aspects] if active_aspects else [0.0]
    net_sentiment = float(np.mean(active_vals))
    sent_conflict = 1 if (any(v > 0.1 for v in active_vals) and any(v < -0.1 for v in active_vals)) else 0
    all_matched_lens = [len(s) for items in matched_clauses_per_aspect.values() for s, _ in items]
    evidence_density = float(min(sum(all_matched_lens) / char_len, 1.0)) if all_matched_lens else clean_token_ratio
    avg_rat_len = float(np.mean(all_matched_lens)) if all_matched_lens else float(char_len / max(asp_count, 1))
    coverage_ratio = float(min(asp_count / 6.0, 1.0))
    consensus_strength = float(abs(net_sentiment))
    temporal_aspect_sentiment = float(recency_weight * net_sentiment)

    return np.array([
        asp_count,
        dominant_aspect,
        net_sentiment,
        sent_conflict,
        evidence_density,
        avg_rat_len,
        overlap_flag,
        aspect_scores["Service"],
        aspect_scores["Facility"],
        aspect_scores["Experience"],
        aspect_scores["Amenity"],
        aspect_scores["Loyalty"],
        aspect_scores["Branding"],
        coverage_ratio,
        recency_weight,
        consensus_strength,
        temporal_aspect_sentiment,
        clean_token_ratio,
        opinion_neg_density,
        ebr_proto_sim
    ], dtype=np.float32)


## 3. Nạp Tập dữ liệu Parquet và Chuẩn hóa Đặc trưng


In [ ]:
parquet_path = next((Path(p) for p in cfg.PARQUET_CANDIDATES if Path(p).exists()), None)
if parquet_path is None and Path("/kaggle/input").exists():
    found_parquets = list(Path("/kaggle/input").rglob("hotel_reviews.parquet"))
    if found_parquets:
        parquet_path = found_parquets[0]

if parquet_path is None:
    raise FileNotFoundError(f"Không tìm thấy hotel_reviews.parquet tại {cfg.PARQUET_CANDIDATES} hoặc trong /kaggle/input/!")

df_master = pd.read_parquet(parquet_path)
if "clean_review_text" not in df_master.columns or "paper1_nltk_text" not in df_master.columns:
    cleaned_pairs = [preprocess_text_paper1(txt) for txt in df_master["review_text"]]
    df_master["clean_review_text"] = [p[0] for p in cleaned_pairs]
    df_master["paper1_nltk_text"] = [p[1] for p in cleaned_pairs]

missing_feats = [c for c in cfg.FEATURE_COLS_20 if c not in df_master.columns]
if missing_feats:
    computed = np.vstack([
        extract_20_features_for_new_review(txt, float(rw))
        for txt, rw in zip(df_master["review_text"], df_master.get("feat_recency_weight", np.ones(len(df_master))))
    ])
    for idx_f, col_name in enumerate(cfg.FEATURE_COLS_20):
        if col_name not in df_master.columns:
            df_master[col_name] = computed[:, idx_f]
else:
    if df_master["feat_dominant_aspect"].dtype == object:
        df_master["feat_dominant_aspect"] = df_master["feat_dominant_aspect"].map(lambda x: ASPECT_ID_MAP.get(str(x), 0)).astype(np.float32)

if "ordinal_rating" not in df_master.columns:
    df_master["ordinal_rating"] = df_master["rating_5"].clip(1, 5).astype(int) - 1

df_train = df_master[df_master["split"] == "train"].reset_index(drop=True)
df_val = df_master[df_master["split"] == "val"].reset_index(drop=True)
df_test = df_master[df_master["split"] == "test"].reset_index(drop=True)

feat_mean = df_train[cfg.FEATURE_COLS_20].values.astype(np.float32).mean(axis=0)
feat_std = df_train[cfg.FEATURE_COLS_20].values.astype(np.float32).std(axis=0)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)
print(f"Tập Train: {len(df_train):,} mẫu | Tập Val: {len(df_val):,} mẫu | Tập Test: {len(df_test):,} mẫu")


## 4. Thiết kế CRAMABSAv2Dataset Huấn luyện Đa nhiệm và Seq2Seq


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)
gen_tokenizer = AutoTokenizer.from_pretrained(cfg.GEN_MODEL_NAME)

class CRAMABSAv2Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, bert_tok, seq2seq_tok, feat_mean: np.ndarray, feat_std: np.ndarray, max_len: int = 256, max_comment_len: int = 48):
        self.df = df.reset_index(drop=True)
        self.bert_tok = bert_tok
        self.seq2seq_tok = seq2seq_tok
        self.feat_mean = feat_mean
        self.feat_std = feat_std
        self.max_len = max_len
        self.max_comment_len = max_comment_len
        self.features_raw = self.df[cfg.FEATURE_COLS_20].values.astype(np.float32)
        self.features_norm = (self.features_raw - self.feat_mean) / self.feat_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        raw_text = unicodedata.normalize("NFC", str(row["review_text"]))
        feat_vec = self.features_norm[idx]

        enc = self.bert_tok(
            raw_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        offsets = enc["offset_mapping"].squeeze(0).numpy()

        ordinal_rating = int(row.get("ordinal_rating", int(row["rating_5"]) - 1))
        ordinal_rating = max(0, min(4, ordinal_rating))

        aspect_sent_labels = torch.zeros(cfg.NUM_ASPECTS, dtype=torch.long)
        aspect_star_labels = torch.full((cfg.NUM_ASPECTS,), ordinal_rating, dtype=torch.long)
        rationale_masks = torch.zeros((cfg.NUM_ASPECTS, self.max_len), dtype=torch.float32)
        gen_span_ids = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)
        gen_span_mask = torch.zeros((cfg.NUM_ASPECTS, self.max_comment_len), dtype=torch.long)

        anns = parse_annotations_safe(row.get("aspect_annotations", []))
        asp_sent_votes = defaultdict(list)
        asp_span_texts = defaultdict(list)

        for ann in anns:
            asp = ann.get("aspect")
            sent = ann.get("sentiment", "Neutral")
            if asp not in ASPECT_ID_MAP or asp == "None":
                continue
            asp_idx = ASPECT_ID_MAP[asp] - 1
            s_char = int(ann.get("start_char", 0))
            e_char = int(ann.get("end_char", 0))
            span_txt = str(ann.get("span_text", "")).strip()
            asp_sent_votes[asp_idx].append(SENTIMENT_TO_ID.get(sent, 2))
            if span_txt:
                asp_span_texts[asp_idx].append(span_txt)

            for t_i, (tok_s, tok_e) in enumerate(offsets):
                if tok_e > tok_s and tok_s < e_char and tok_e > s_char:
                    rationale_masks[asp_idx, t_i] = 1.0

        for asp_idx, votes in asp_sent_votes.items():
            maj_sent = Counter(votes).most_common(1)[0][0]
            aspect_sent_labels[asp_idx] = maj_sent

            joined_span = " ".join(asp_span_texts[asp_idx]).strip()
            span_raw_toks, _ = nltk_negation_aware_preprocess(joined_span if joined_span else raw_text)
            has_hedge = any(t in HEDGING_MODIFIERS for t in span_raw_toks)

            if maj_sent == 3:
                asp_star = 3 if has_hedge else max(3, ordinal_rating)
            elif maj_sent == 2:
                asp_star = 2
            else:
                asp_star = 1 if has_hedge else min(1, ordinal_rating)
            aspect_star_labels[asp_idx] = asp_star

            if joined_span:
                t_enc = self.seq2seq_tok(
                    joined_span,
                    max_length=self.max_comment_len,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                gen_span_ids[asp_idx] = t_enc["input_ids"].squeeze(0)
                gen_span_mask[asp_idx] = t_enc["attention_mask"].squeeze(0)

        aspect_corn_targets = torch.zeros((cfg.NUM_ASPECTS, cfg.NUM_STARS - 1), dtype=torch.float32)
        for a_i in range(cfg.NUM_ASPECTS):
            for k in range(cfg.NUM_STARS - 1):
                if aspect_star_labels[a_i] > k:
                    aspect_corn_targets[a_i, k] = 1.0

        corn_targets = torch.tensor(
            [1.0 if ordinal_rating > k else 0.0 for k in range(cfg.NUM_STARS - 1)],
            dtype=torch.float32
        )

        return {
            "review_id": int(row["review_id"]),
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "dense_features": torch.tensor(feat_vec, dtype=torch.float32),
            "aspect_sent_labels": aspect_sent_labels,
            "aspect_star_labels": aspect_star_labels,
            "aspect_corn_targets": aspect_corn_targets,
            "rationale_masks": rationale_masks,
            "gen_span_ids": gen_span_ids,
            "gen_span_mask": gen_span_mask,
            "ordinal_rating": torch.tensor(ordinal_rating, dtype=torch.long),
            "corn_targets": corn_targets
        }

train_dataset = CRAMABSAv2Dataset(df_train, tokenizer, gen_tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)
val_dataset = CRAMABSAv2Dataset(df_val, tokenizer, gen_tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)
test_dataset = CRAMABSAv2Dataset(df_test, tokenizer, gen_tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
print(f"Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")


## 5. Kiến trúc CRAM-ABSA v2 Tích hợp Neural Seq2Seq Generator (Flan-T5 Soft-Prompt Conditioning)


In [ ]:
class CRAM_ABSA_v2_MultiAspectModel(nn.Module):
    def __init__(self, cfg: TrainConfigV2):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.num_sent_classes = cfg.NUM_SENTIMENT_CLASSES
        self.hidden_dim = cfg.HIDDEN_DIM
        self.max_comment_len = getattr(cfg, "MAX_COMMENT_LEN", 48)
        self.num_prompt_tokens = getattr(cfg, "NUM_SOFT_PROMPT_TOKENS", 4)
        self.canonical_aspects = list(cfg.CANONICAL_ASPECTS)

        self.bert = AutoModel.from_pretrained(cfg.BACKBONE_NAME)
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.GEN_MODEL_NAME)
        self.gen_d_model = self.gen_model.config.d_model

        for param in self.gen_model.parameters():
            param.requires_grad = False

        self.feature_proj = nn.Sequential(
            nn.Linear(cfg.NUM_FEATURES, cfg.PROJ_DIM),
            nn.LayerNorm(cfg.PROJ_DIM),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(cfg.PROJ_DIM, cfg.HIDDEN_DIM),
            nn.LayerNorm(cfg.HIDDEN_DIM)
        )
        self.residual_gate = nn.Parameter(torch.tensor(0.30))

        self.ebr_aspect_prototypes = nn.Parameter(torch.randn(cfg.NUM_ASPECTS, cfg.HIDDEN_DIM) * 0.02)
        self.aspect_cross_attn = nn.MultiheadAttention(
            embed_dim=cfg.HIDDEN_DIM,
            num_heads=cfg.ATTN_HEADS,
            dropout=0.1,
            batch_first=True
        )
        self.rationale_proj = nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM)
        self.rationale_bias = nn.Parameter(torch.zeros(cfg.NUM_ASPECTS, 1))

        self.aspect_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
                nn.LayerNorm(cfg.PROJ_DIM),
                nn.GELU(),
                nn.Dropout(0.20),
                nn.Linear(cfg.PROJ_DIM, cfg.NUM_SENTIMENT_CLASSES)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])

        self.aspect_star_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
                nn.LayerNorm(cfg.PROJ_DIM),
                nn.GELU(),
                nn.Dropout(0.20),
                nn.Linear(cfg.PROJ_DIM, cfg.NUM_STARS - 1)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])

        self.corn_head = nn.Sequential(
            nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
            nn.LayerNorm(cfg.PROJ_DIM),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(cfg.PROJ_DIM, cfg.NUM_STARS - 1)
        )

        self.aspect_soft_prompt = nn.Sequential(
            nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
            nn.GELU(),
            nn.Linear(cfg.PROJ_DIM, self.num_prompt_tokens * self.gen_d_model),
            nn.LayerNorm(self.num_prompt_tokens * self.gen_d_model)
        )

    @torch.no_grad()
    def initialize_semantic_prototypes(self, bert_tokenizer, aspect_keywords_dict: Dict[str, set]):
        device = next(self.bert.parameters()).device
        for a_idx, asp in enumerate(self.canonical_aspects):
            kw_text = f"{asp}: " + " ".join(sorted(list(aspect_keywords_dict.get(asp, {asp.lower()}))))
            enc = bert_tokenizer(kw_text, return_tensors="pt", truncation=True, max_length=64).to(device)
            out = self.bert(**enc)
            mean_vec = out.last_hidden_state.mean(dim=1).squeeze(0)
            self.ebr_aspect_prototypes.data[a_idx].copy_(F.normalize(mean_vec, p=2, dim=-1) * 0.5)

    def forward(self, input_ids, attention_mask, dense_features, gen_span_ids=None, gen_span_mask=None):
        bsz, seq_len = input_ids.size()
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_hidden = bert_out.last_hidden_state
        cls_hidden = seq_hidden[:, 0, :]

        feat_emb = self.feature_proj(dense_features)
        conditioned_seq = seq_hidden + self.residual_gate * feat_emb.unsqueeze(1)
        conditioned_cls = cls_hidden + feat_emb

        queries = self.ebr_aspect_prototypes.unsqueeze(0).expand(bsz, -1, -1)
        key_padding_mask = (attention_mask == 0)
        cross_ctx, cross_attn_weights = self.aspect_cross_attn(
            query=queries,
            key=conditioned_seq,
            value=conditioned_seq,
            key_padding_mask=key_padding_mask,
            need_weights=True
        )

        token_proj = self.rationale_proj(conditioned_seq)
        proto_norm = F.normalize(self.ebr_aspect_prototypes, p=2, dim=-1)
        token_norm = F.normalize(token_proj, p=2, dim=-1)
        bilinear_scores = torch.einsum("bld,ad->bal", token_proj, self.ebr_aspect_prototypes) / math.sqrt(self.hidden_dim)
        cosine_scores = torch.einsum("bld,ad->bal", token_norm, proto_norm)
        rationale_logits = bilinear_scores + cosine_scores + torch.log(cross_attn_weights.clamp(min=1e-6)) + self.rationale_bias

        pad_mask = attention_mask.unsqueeze(1).float()
        rationale_probs = torch.sigmoid(rationale_logits) * pad_mask

        denom = rationale_probs.sum(dim=-1, keepdim=True).clamp(min=1e-4)
        masked_pooled = 0.5 * (torch.einsum("bal,bld->bad", rationale_probs, conditioned_seq) / denom) + 0.5 * cross_ctx

        aspect_logits_list = []
        aspect_star_logits_list = []
        soft_prompts_list = []
        for a_idx in range(self.num_aspects):
            asp_ctx = torch.cat([masked_pooled[:, a_idx, :], feat_emb], dim=-1)
            aspect_logits_list.append(self.aspect_classifiers[a_idx](asp_ctx))
            aspect_star_logits_list.append(self.aspect_star_heads[a_idx](asp_ctx))
            sp = self.aspect_soft_prompt(asp_ctx).view(bsz, self.num_prompt_tokens, self.gen_d_model)
            soft_prompts_list.append(sp)

        aspect_logits = torch.stack(aspect_logits_list, dim=1)
        aspect_star_logits = torch.stack(aspect_star_logits_list, dim=1)
        soft_prompts = torch.stack(soft_prompts_list, dim=1)

        global_aspect_ctx = masked_pooled.mean(dim=1)
        corn_input = torch.cat([conditioned_cls, global_aspect_ctx], dim=-1)
        corn_logits = self.corn_head(corn_input)

        gen_loss = torch.zeros(1, device=input_ids.device, dtype=torch.float32)
        if gen_span_ids is not None and gen_span_mask is not None:
            flat_prompts = soft_prompts.view(bsz * self.num_aspects, self.num_prompt_tokens, self.gen_d_model)
            flat_ids = gen_span_ids.view(bsz * self.num_aspects, -1)
            flat_mask = gen_span_mask.view(bsz * self.num_aspects, -1)
            valid_rows = flat_mask.sum(dim=-1) > 1
            if valid_rows.any():
                active_prompts = flat_prompts[valid_rows].float()
                active_ids = flat_ids[valid_rows]
                active_mask = flat_mask[valid_rows]
                with torch.no_grad():
                    t5_tok_emb = self.gen_model.get_input_embeddings()(active_ids).float()
                    mask_f = active_mask.unsqueeze(-1).float()
                    target_vec = (t5_tok_emb * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
                pred_vec = active_prompts.mean(dim=1)
                cos_dist = 1.0 - F.cosine_similarity(pred_vec, target_vec, dim=-1).mean()
                l1_dist = F.smooth_l1_loss(pred_vec, target_vec)
                gen_loss = (cos_dist + l1_dist).view(1)

        return {
            "aspect_logits": aspect_logits,
            "aspect_star_logits": aspect_star_logits,
            "rationale_logits": rationale_logits,
            "rationale_probs": rationale_probs,
            "corn_logits": corn_logits,
            "gen_loss": gen_loss,
            "soft_prompts": soft_prompts,
            "aspect_embeddings": masked_pooled,
            "seq_hidden": conditioned_seq
        }

    @torch.no_grad()
    def generate_aspect_commentary(
        self,
        aspect_idx: int,
        aspect_name: str,
        sentiment_label: str,
        star_val: int,
        review_text: str,
        outputs: Dict[str, torch.Tensor],
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        bert_tokenizer,
        seq2seq_tokenizer
    ) -> str:
        device = input_ids.device
        rat_prob = outputs["rationale_probs"][0, aspect_idx]
        valid_len = int(attention_mask[0].sum().item())
        tok_ids = input_ids[0, :valid_len].cpu().tolist()
        scores = rat_prob[:valid_len].cpu().numpy()

        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", review_text.strip()) if s.strip()]
        if len(sentences) > 1:
            scored_sents = []
            for s in sentences:
                s_ids = set(bert_tokenizer.encode(s, add_special_tokens=False))
                hit_scores = [scores[i] for i in range(1, valid_len - 1) if tok_ids[i] in s_ids]
                mean_sc = float(np.mean(hit_scores)) if hit_scores else 0.0
                scored_sents.append((mean_sc, s))
            scored_sents.sort(key=lambda x: x[0], reverse=True)
            focused_evidence = " ".join([s for _, s in scored_sents[:2]])
        else:
            focused_evidence = review_text.strip()

        prompt_str = (
            f"Paraphrase and summarize the guest's opinion on the hotel {aspect_name} "
            f"({sentiment_label}, {star_val}/5 stars) into a short third-person analytical comment: {focused_evidence}"
        )
        p_enc = seq2seq_tokenizer(
            prompt_str,
            max_length=192,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        tok_embeds = self.gen_model.get_input_embeddings()(p_enc["input_ids"]).float()
        soft_p = outputs["soft_prompts"][0:1, aspect_idx, :, :].float()
        combined_embeds = torch.cat([0.20 * soft_p, tok_embeds], dim=1)
        prompt_mask = torch.ones((1, soft_p.size(1)), dtype=p_enc["attention_mask"].dtype, device=device)
        combined_mask = torch.cat([prompt_mask, p_enc["attention_mask"]], dim=1)

        enc_out = self.gen_model.encoder(
            inputs_embeds=combined_embeds,
            attention_mask=combined_mask
        )

        gen_ids = self.gen_model.generate(
            encoder_outputs=enc_out,
            attention_mask=combined_mask,
            max_new_tokens=self.max_comment_len,
            min_length=8,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.25,
            early_stopping=True
        )
        return seq2seq_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()


## 6. Hàm Mất mát Đa nhiệm CRAMABSAv2Loss


In [ ]:
class CRAMABSAv2Loss(nn.Module):
    def __init__(self, cfg: TrainConfigV2, class_weights_corn: torch.Tensor):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("corn_weights", class_weights_corn)
        self.aspect_ce = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)

    def compute_corn_loss(self, corn_logits: torch.Tensor, corn_targets: torch.Tensor, ordinal_rating: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(corn_logits, corn_targets, reduction="none")
        sample_w = self.corn_weights[ordinal_rating].unsqueeze(-1)
        return (bce * sample_w).mean()

    def compute_overlap_penalty(self, rationale_probs: torch.Tensor) -> torch.Tensor:
        norm_masks = F.normalize(rationale_probs, p=2, dim=-1, eps=1e-6)
        gram_matrix = torch.bmm(norm_masks, norm_masks.transpose(1, 2))
        eye = torch.eye(rationale_probs.size(1), device=rationale_probs.device).unsqueeze(0)
        off_diag = gram_matrix * (1.0 - eye)
        num_pairs = rationale_probs.size(1) * (rationale_probs.size(1) - 1)
        return off_diag.sum(dim=(1, 2)).mean() / max(num_pairs, 1)

    def forward(self, outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        aspect_logits = outputs["aspect_logits"]
        aspect_star_logits = outputs["aspect_star_logits"]
        rationale_logits = outputs["rationale_logits"]
        rationale_probs = outputs["rationale_probs"]
        corn_logits = outputs["corn_logits"]
        gen_loss = outputs.get("gen_loss", torch.zeros(1, device=aspect_logits.device)).mean()

        aspect_targets = batch["aspect_sent_labels"]
        aspect_corn_targets = batch["aspect_corn_targets"]
        rationale_targets = batch["rationale_masks"]
        corn_targets = batch["corn_targets"]
        ordinal_rating = batch["ordinal_rating"]
        pad_mask = batch["attention_mask"].unsqueeze(1).float()

        loss_aspect = self.aspect_ce(aspect_logits.view(-1, self.cfg.NUM_SENTIMENT_CLASSES), aspect_targets.view(-1))
        loss_aspect_star = F.binary_cross_entropy_with_logits(aspect_star_logits, aspect_corn_targets)
        loss_corn = self.compute_corn_loss(corn_logits, corn_targets, ordinal_rating)
        loss_overlap = self.compute_overlap_penalty(rationale_probs)

        pos_w = torch.tensor(4.0, device=rationale_logits.device)
        bce_rat = F.binary_cross_entropy_with_logits(
            rationale_logits, rationale_targets, pos_weight=pos_w, reduction="none"
        )
        loss_rationale = (bce_rat * pad_mask).sum() / pad_mask.sum().clamp(min=1.0)

        total_loss = (
            loss_aspect
            + self.cfg.BETA_CORN * (loss_corn + loss_aspect_star)
            + self.cfg.BETA_OVERLAP * loss_overlap
            + self.cfg.BETA_RATIONALE * loss_rationale
            + self.cfg.BETA_COMMENTARY * gen_loss
        )
        return {
            "total_loss": total_loss,
            "loss_aspect": loss_aspect.detach(),
            "loss_corn": loss_corn.detach(),
            "loss_overlap": loss_overlap.detach(),
            "loss_rationale": loss_rationale.detach(),
            "loss_commentary": gen_loss.detach()
        }

def decode_corn_predictions(corn_logits: torch.Tensor) -> torch.Tensor:
    probs = torch.sigmoid(corn_logits)
    cum_probs = torch.cumprod(probs, dim=-1)
    return (cum_probs >= 0.5).sum(dim=-1)

star_counts = df_train["ordinal_rating"].value_counts().sort_index()
total_tr = len(df_train)
corn_w = torch.tensor(
    [math.sqrt(total_tr / (5.0 * max(star_counts.get(k, 1), 1))) for k in range(5)],
    dtype=torch.float32,
    device=DEVICE
)


## 7. Huấn luyện Mô hình CRAM-ABSA v2 và Lưu Trọng số modelv2.pt


In [ ]:
from contextlib import nullcontext

def get_amp_context(enabled: bool):
    if not enabled or DEVICE.type != "cuda":
        return nullcontext()
    try:
        return torch.amp.autocast("cuda", dtype=torch.float16, enabled=True)
    except Exception:
        return torch.cuda.amp.autocast(enabled=True)

def create_grad_scaler(enabled: bool):
    if not enabled or DEVICE.type != "cuda":
        try:
            return torch.amp.GradScaler("cuda", enabled=False)
        except Exception:
            return torch.cuda.amp.GradScaler(enabled=False)
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)

base_model = CRAM_ABSA_v2_MultiAspectModel(cfg).to(DEVICE)
base_model.initialize_semantic_prototypes(tokenizer, ASPECT_KEYWORDS)
if NUM_GPUS > 1:
    model = nn.DataParallel(base_model, device_ids=list(range(NUM_GPUS)))
else:
    model = base_model

criterion = CRAMABSAv2Loss(cfg, corn_w).to(DEVICE)

raw_mod = model.module if isinstance(model, nn.DataParallel) else model
optimizer_grouped_parameters = [
    {"params": [p for p in raw_mod.bert.parameters() if p.requires_grad], "lr": cfg.LR_BACKBONE, "weight_decay": cfg.WEIGHT_DECAY},
    {"params": [p for n, p in raw_mod.named_parameters() if not n.startswith("bert.") and not n.startswith("gen_model.") and p.requires_grad], "lr": cfg.LR_HEADS, "weight_decay": cfg.WEIGHT_DECAY}
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
total_steps = max(1, math.ceil(len(train_loader) / cfg.GRAD_ACCUM_STEPS) * cfg.EPOCHS)
warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = create_grad_scaler(cfg.USE_AMP_FP16)

@torch.no_grad()
def evaluate_cram_absa_v2(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_star_preds, all_star_trues = [], []
    all_asp_preds, all_asp_trues = [], []
    rat_inter, rat_pred_sum, rat_true_sum = 0.0, 0.0, 0.0
    overlap_rates = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        dense_features = batch["dense_features"].to(DEVICE, non_blocking=True)
        batch_gpu = {k: (v.to(DEVICE, non_blocking=True) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

        with get_amp_context(cfg.USE_AMP_FP16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                dense_features=dense_features,
                gen_span_ids=batch_gpu.get("gen_span_ids"),
                gen_span_mask=batch_gpu.get("gen_span_mask")
            )
            losses = criterion(outputs, batch_gpu)

        total_loss += losses["total_loss"].item()
        star_preds = decode_corn_predictions(outputs["corn_logits"])
        all_star_preds.extend(star_preds.cpu().numpy())
        all_star_trues.extend(batch["ordinal_rating"].numpy())

        asp_preds = outputs["aspect_logits"].argmax(dim=-1)
        all_asp_preds.append(asp_preds.cpu().numpy())
        all_asp_trues.append(batch["aspect_sent_labels"].numpy())

        pred_mask = (outputs["rationale_probs"] >= 0.5).float()
        true_mask = batch_gpu["rationale_masks"]
        rat_inter += (pred_mask * true_mask).sum().item()
        rat_pred_sum += pred_mask.sum().item()
        rat_true_sum += true_mask.sum().item()

        active_asp_per_tok = pred_mask.sum(dim=1)
        leaked_toks = (active_asp_per_tok > 1.0).float().sum().item()
        active_toks = (active_asp_per_tok >= 1.0).float().sum().item()
        overlap_rates.append(leaked_toks / max(active_toks, 1.0))

    y_true_star = np.array(all_star_trues)
    y_pred_star = np.array(all_star_preds)
    acc_at_0 = accuracy_score(y_true_star, y_pred_star) * 100.0

    per_star_mae = [
        np.abs(y_true_star[y_true_star == k] - y_pred_star[y_true_star == k]).mean()
        for k in range(5) if np.any(y_true_star == k)
    ]
    macro_mae = float(np.mean(per_star_mae)) if per_star_mae else 0.0

    Y_asp_true = np.vstack(all_asp_trues)
    Y_asp_pred = np.vstack(all_asp_preds)
    aspect_f1s = [
        f1_score(Y_asp_true[:, a], Y_asp_pred[:, a], average="macro", zero_division=0) * 100.0
        for a in range(cfg.NUM_ASPECTS)
    ]
    macro_aspect_f1 = float(np.mean(aspect_f1s))
    rationale_f1 = (2.0 * rat_inter / max(rat_pred_sum + rat_true_sum, 1.0)) * 100.0
    leakage_rate = float(np.mean(overlap_rates)) * 100.0

    return {
        "val_loss": total_loss / max(len(loader), 1),
        "acc_at_0": acc_at_0,
        "macro_mae": macro_mae,
        "aspect_macro_f1": macro_aspect_f1,
        "rationale_f1": rationale_f1,
        "leakage_rate": leakage_rate,
        "per_aspect_f1": dict(zip(cfg.CANONICAL_ASPECTS, [round(x, 2) for x in aspect_f1s]))
    }

history = []
best_score = -float("inf")

print("Bắt đầu huấn luyện mô hình CRAM-ABSA v2...")
for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    ep_start = time.time()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader, 1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        dense_features = batch["dense_features"].to(DEVICE, non_blocking=True)
        batch_gpu = {k: (v.to(DEVICE, non_blocking=True) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

        with get_amp_context(cfg.USE_AMP_FP16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                dense_features=dense_features,
                gen_span_ids=batch_gpu.get("gen_span_ids"),
                gen_span_mask=batch_gpu.get("gen_span_mask")
            )
            losses = criterion(outputs, batch_gpu)
            loss = losses["total_loss"] / cfg.GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += losses["total_loss"].item()

        if step % cfg.GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    val_metrics = evaluate_cram_absa_v2(model, val_loader, criterion)
    ep_time = time.time() - ep_start
    vram_used = sum(torch.cuda.max_memory_allocated(i) / (1024 ** 3) for i in range(NUM_GPUS)) if NUM_GPUS > 0 else 0.0

    composite_score = val_metrics["aspect_macro_f1"] + val_metrics["acc_at_0"] - 20.0 * val_metrics["macro_mae"]
    is_best = composite_score > best_score
    if is_best:
        best_score = composite_score
        raw_state = (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
        ckpt_payload = {
            "epoch": epoch,
            "model_name": "CRAM-ABSA v2 (modelv2.pt)",
            "architecture": "cram_absa_v2",
            "gen_model_name": cfg.GEN_MODEL_NAME,
            "model_state_dict": raw_state,
            "feat_mean": feat_mean,
            "feat_std": feat_std,
            "feature_cols_20": cfg.FEATURE_COLS_20,
            "canonical_aspects": cfg.CANONICAL_ASPECTS,
            "val_metrics": val_metrics
        }
        torch.save(ckpt_payload, cfg.MODEL_SAVE_PATH)
        if cfg.LOCAL_MODEL_SAVE_PATH.resolve() != cfg.MODEL_SAVE_PATH.resolve():
            shutil.copy2(cfg.MODEL_SAVE_PATH, cfg.LOCAL_MODEL_SAVE_PATH)

    epoch_record = {
        "epoch": epoch,
        "train_loss": round(running_loss / max(len(train_loader), 1), 4),
        **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_metrics.items()},
        "vram_gb": round(vram_used, 2),
        "elapsed_s": round(ep_time, 1)
    }
    history.append(epoch_record)
    star_flag = "★ [BEST -> modelv2.pt]" if is_best else ""
    print(
        f"Epoch {epoch:02d}/{cfg.EPOCHS:02d} ({ep_time:.1f}s | VRAM: {vram_used:.2f}GB) | "
        f"Train Loss: {epoch_record['train_loss']:.4f} | Val Acc@0: {val_metrics['acc_at_0']:.2f}% | "
        f"6-Aspect F1: {val_metrics['aspect_macro_f1']:.2f}% | Macro-MAE: {val_metrics['macro_mae']:.4f} | "
        f"Rat-F1: {val_metrics['rationale_f1']:.2f}% | Leakage: {val_metrics['leakage_rate']:.2f}% {star_flag}"
    )

with open(cfg.HISTORY_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2, ensure_ascii=False)
print(f"\nHoàn tất huấn luyện! File model v2 đã được lưu với tên: {cfg.MODEL_SAVE_PATH}")


## 8. Suy luận Review Mới 100% Bằng Trọng số Neural


In [ ]:
@torch.no_grad()
def generate_structured_review_output_v2(
    new_review_text: str,
    model: nn.Module,
    bert_tokenizer,
    seq2seq_tokenizer,
    feat_mean: Optional[np.ndarray] = None,
    feat_std: Optional[np.ndarray] = None
) -> Dict[str, Any]:
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.eval()

    raw_text = unicodedata.normalize("NFC", str(new_review_text).strip())
    raw_feat_20 = extract_20_features_for_new_review(raw_text, recency_weight=1.0)
    if feat_mean is not None and feat_std is not None:
        norm_feat_20 = (raw_feat_20 - feat_mean) / np.where(feat_std < 1e-6, 1.0, feat_std)
    else:
        norm_feat_20 = raw_feat_20

    enc = bert_tokenizer(
        raw_text,
        max_length=cfg.MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    dense_tensor = torch.tensor(norm_feat_20, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    outputs = raw_model(input_ids=input_ids, attention_mask=attention_mask, dense_features=dense_tensor)
    pred_global_star = int(decode_corn_predictions(outputs["corn_logits"]).item()) + 1
    asp_probs = F.softmax(outputs["aspect_logits"].squeeze(0), dim=-1).cpu().numpy()
    asp_stars = decode_corn_predictions(outputs["aspect_star_logits"].squeeze(0)).cpu().numpy() + 1

    mentioned_aspects = []
    aspect_outputs = {}

    for a_idx, asp in enumerate(cfg.CANONICAL_ASPECTS):
        pred_cls = int(np.argmax(asp_probs[a_idx]))
        if pred_cls != 0:
            mentioned_aspects.append(asp)
            star_val = int(asp_stars[a_idx])
            sent_label = ID_TO_SENTIMENT[pred_cls]
            gen_text = raw_model.generate_aspect_commentary(
                aspect_idx=a_idx,
                aspect_name=asp,
                sentiment_label=sent_label,
                star_val=star_val,
                review_text=raw_text,
                outputs=outputs,
                input_ids=input_ids,
                attention_mask=attention_mask,
                bert_tokenizer=bert_tokenizer,
                seq2seq_tokenizer=seq2seq_tokenizer
            )
            aspect_outputs[asp] = {
                "mentioned": True,
                "stars": star_val,
                "sentiment": sent_label,
                "text": f"{star_val}/5 ★ ({sent_label}) - {gen_text}"
            }
        else:
            aspect_outputs[asp] = {
                "mentioned": False,
                "stars": None,
                "sentiment": "Not Mentioned",
                "text": ""
            }

    aspect_list_str = ", ".join(mentioned_aspects) if mentioned_aspects else "None"
    lines = [f"- Aspect: {aspect_list_str}"]
    for asp in cfg.CANONICAL_ASPECTS:
        info = aspect_outputs[asp]
        if info["mentioned"]:
            lines.append(f"- {asp}: {info['text']}")
        else:
            lines.append(f"- {asp}:")

    return {
        "Aspect": mentioned_aspects,
        "overall_star": pred_global_star,
        "aspects": aspect_outputs,
        "formatted_text": "\n".join(lines)
    }

def load_trained_model_v2_and_predict(new_review_text: str, checkpoint_path: Optional[str] = None) -> str:
    ckpt_file = Path(checkpoint_path) if checkpoint_path else cfg.MODEL_SAVE_PATH
    if not ckpt_file.exists() and cfg.LOCAL_MODEL_SAVE_PATH.exists():
        ckpt_file = cfg.LOCAL_MODEL_SAVE_PATH
    if not ckpt_file.exists():
        raise FileNotFoundError(f"Không tìm thấy file mô hình tại: {ckpt_file}")

    ckpt = torch.load(ckpt_file, map_location=DEVICE, weights_only=False)
    infer_model = CRAM_ABSA_v2_MultiAspectModel(cfg).to(DEVICE)
    infer_model.load_state_dict(ckpt["model_state_dict"], strict=False)

    f_mean = ckpt.get("feat_mean", globals().get("feat_mean", None))
    f_std = ckpt.get("feat_std", globals().get("feat_std", None))
    b_tok = globals().get("tokenizer", None) or AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)
    g_tok = globals().get("gen_tokenizer", None) or AutoTokenizer.from_pretrained(cfg.GEN_MODEL_NAME)

    res = generate_structured_review_output_v2(
        new_review_text=new_review_text,
        model=infer_model,
        bert_tokenizer=b_tok,
        seq2seq_tokenizer=g_tok,
        feat_mean=f_mean,
        feat_std=f_std
    )
    return res["formatted_text"]
